# 🐾 Deep Learning in the Kalahari
## *Understanding Deep Learning*, chapters 1–8 in one hour

**Transition Bio · ML Reading Group**

A one-hour, hands-on tour of chapters 1–8 of Simon J.D. Prince's **_Understanding Deep Learning_** (MIT Press; free at [udlbook.github.io](https://udlbook.github.io/udlbook/)). We follow the book's notation, square brackets and all: $\mathbf{y} = \mathbf{f}[\mathbf{x}, \boldsymbol\phi]$.

### ⏱️ The hour
| Time | What |
|---|---|
| 0–5 min | Slides: welcome and roadmap |
| 5–55 min | Chapter by chapter: a problem slide, then **this notebook** to solve it, then the solution slide |
| 55–60 min | Wrap-up and discussion |

### 🦦 The story
You've just joined a field station in the Kalahari that studies a wild meerkat mob. The station has a lot of data and no deep-learning expert, so you're it. All datasets are **synthetic**: they're generated right here, so every answer can be checked automatically.

### 🧭 How it works
| Symbol | Meaning |
|---|---|
| 📖 | Recap of the chapter's maths. Skim it; it's there to come back to |
| 💻 | Demo code: just run it and look at the plot |
| 🧩 | **Core puzzle**: fill in the `TODO`, then run the 🧪 check cell |
| 🏆 | Challenge with bronze / silver / gold targets (do these if you have time) |
| 🔑 | Hidden solution. Have a go first; run it if stuck so later cells still work |
| ⚙️ | Ready-made helper: just run it |

> **Tip:** start with *Runtime → Run all*. Every TODO will report "still a TODO" until you fill it in, and everything else just works. Then go top to bottom through the puzzles.

### 🗺️ The core path (≈ 40 min)
| Part | Chapter | Puzzles | Time |
|---|---|---|---|
| 1 | Introduction | read only | 1 min |
| 2 | Supervised learning | 2.1 gradient of the least squares loss | 3 min |
| 3 | Shallow networks | 3.1 joints & slopes · 🏆 dig the burrow | 8 min |
| 4 | Deep networks | 4.1 predict the folds | 3 min |
| 5 | Loss functions | 5.3 BCE gradient · 5.4 stable softmax · 5.6 which way is the burrow? | 9 min |
| 6 | Fitting models | 6.4 momentum / Nesterov / Adam · 🏆 race to the burrow | 8 min |
| 7 | Gradients & init | 7.1 backprop by hand | 6 min |
| 8 | Measuring performance | 8.1 · 8.4 (quick questions) | 2 min |

⏱ **Short on time?** Skip the 🏆 challenges first. At the end, `progress()` shows what you've solved; it's just for you, and nothing is recorded or shared.

*Book figures and problem numbers are cited as e.g. "Fig 3.3" and "P3.8".*

In [ ]:
#@title 🐾 Setup: run me first! (plotting style, score keeper, referee)
import math, itertools, time, warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
warnings.filterwarnings("ignore", category=RuntimeWarning)
np.set_printoptions(precision=4, suppress=True)

# Transition Bio palette
TB = dict(green="#008759", lgreen="#73AE63", blue="#1366AB", lblue="#73B0E6",
          purple="#8F64B7", red="#CF1D10", orange="#EF8E00", grey="#6B6B6B", sand="#E9D8B4")
plt.rcParams.update({
    "figure.dpi": 110, "figure.figsize": (6, 4), "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25,
    "axes.prop_cycle": plt.cycler(color=[TB["green"], TB["blue"], TB["orange"],
                                         TB["purple"], TB["red"], TB["lblue"], TB["lgreen"]]),
    "font.size": 10.5, "axes.titleweight": "bold"})

# ---------------- progress tracker (private to you; nothing is shared) ----------------
_SOLVED, _PEEKED, _SEEN = set(), set(), set()

def peek(pid):
    _PEEKED.add(pid)

def check(pid, stars, test, ok_msg="", hint=""):
    """Run `test` (a zero-arg function returning True/False) and report the result."""
    _SEEN.add(pid)
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"🕳️  Puzzle {pid}: still a TODO. Fill it in and re-run (or run the 🔑 cell).")
        return
    except Exception as e:
        print(f"💥 Puzzle {pid} raised {type(e).__name__}: {e}")
        if hint: print("   💡 Hint:", hint)
        return
    if ok:
        _SOLVED.add(pid)
        print(f"✅ Puzzle {pid} solved!  {ok_msg}")
    else:
        print(f"❌ Puzzle {pid}: not quite.", ("💡 " + hint) if hint else "")

def progress():
    own = sorted(_SOLVED - _PEEKED); helped = sorted(_SOLVED & _PEEKED); todo = sorted(_SEEN - _SOLVED)
    print(f"🐾 {len(_SOLVED)} of {len(_SEEN)} puzzles solved "
          f"({len(own)} on your own, {len(helped)} after opening the solution).")
    if todo: print("   Still to do:", ", ".join(todo))

def close(a, b, tol=1e-6):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return a.shape == b.shape and np.allclose(a, b, atol=tol, rtol=tol)

def num_grad(fn, params, eps=1e-6):
    """Central finite-difference gradient of scalar fn(params) w.r.t. a 1D array."""
    params = np.array(params, float); g = np.zeros_like(params)
    for i in range(len(params)):
        d = np.zeros_like(params); d[i] = eps
        g[i] = (fn(params + d) - fn(params - d)) / (2 * eps)
    return g

def medal(score, bronze, silver, gold, higher_is_better=False):
    s = score if higher_is_better else -score
    b, si, g = (bronze, silver, gold) if higher_is_better else (-bronze, -silver, -gold)
    if s >= g: return "🥇 GOLD"
    if s >= si: return "🥈 SILVER"
    if s >= b: return "🥉 BRONZE"
    return "🐣 no medal yet: keep digging"

try:
    import torch
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
print("Setup complete 🐾  | numpy", np.__version__, "| torch available:", HAS_TORCH)

---
# Part 1 · Introduction (Chapter 1)
### 📖 Recap
The book sorts machine learning into three families:

**Supervised learning** learns a mapping from an input $\mathbf{x}$ to an output $\mathbf{y}$ using example pairs $\{\mathbf{x}_i,\mathbf{y}_i\}$. The model is a *family* of equations $\mathbf{y}=\mathbf{f}[\mathbf{x},\boldsymbol\phi]$, and the parameters $\boldsymbol\phi$ pick one member of the family.
- **Regression**: continuous outputs (univariate or multivariate).
- **Classification**: discrete outputs (binary or multiclass).
- Inputs can be tabular, or *structured* (audio, images, text, molecules). Structured inputs are high-dimensional and their elements are related to one another.

**Unsupervised learning** learns from $\{\mathbf{x}_i\}$ alone, with no labels.
- **Generative models** learn to *synthesize* new examples. They can be probabilistic (giving $Pr(\mathbf{x})$) or not.
- **Latent variables** $\mathbf{z}$ capture the underlying structure, so that $\mathbf{x}$ can be generated from a simpler $\mathbf{z}$.

**Reinforcement learning**: an agent takes **actions** in an environment, changes its **state**, and receives **rewards**. The hard part is *temporal credit assignment*: working out which of many past actions earned a delayed reward.

Chapter 1 ends with **ethics**. Bias and fairness, explainability, weaponization, concentration of power and environmental cost all matter at TBio too. What data we train on shapes what the model "believes".

---
# Part 2 · Supervised learning (Chapter 2): 🍼 the Great Pup Weigh-in
### 📖 Recap
Every morning the station weighs the pups on a kitchen scale (with a boiled-egg bribe). We want to predict **mass gained (g)** from **hours spent foraging** the day before.

**Model** (1D linear regression, eq 2.4):
$$y = \mathrm{f}[x,\boldsymbol\phi] = \phi_0 + \phi_1 x .$$

**Loss**: the least squares loss (eq 2.5) measures the mismatch over the training set:
$$L[\boldsymbol\phi] = \sum_{i=1}^{I}\big(\mathrm{f}[x_i,\boldsymbol\phi]-y_i\big)^2=\sum_{i=1}^{I}\big(\phi_0+\phi_1x_i-y_i\big)^2 .$$

**Training** means finding $\hat{\boldsymbol\phi}=\underset{\boldsymbol\phi}{\mathrm{argmin}}\, L[\boldsymbol\phi]$. **Testing** means measuring performance on *new* data to see how well the model **generalizes** (underfitting / overfitting; much more on this in Part 8).

In [ ]:
# 💻 The weigh-in data (synthetic, but the pups are very real in spirit)
rng = np.random.default_rng(2)
I = 20
hours = np.sort(rng.uniform(0.5, 7.5, I))                 # x: foraging hours
gain = 1.5 + 2.2 * hours + rng.normal(0, 2.0, I)          # y: grams gained (true phi = [1.5, 2.2])

def f_linear(x, phi0, phi1):
    return phi0 + phi1 * x

def loss_ls(phi0, phi1, x, y):
    return np.sum((f_linear(x, phi0, phi1) - y) ** 2)

plt.scatter(hours, gain, color=TB["orange"], zorder=3, label="pups")
xx = np.linspace(0, 8, 100)
plt.plot(xx, f_linear(xx, 0, 3), "--", color=TB["grey"], label=r"guess $\phi=[0,3]$")
plt.xlabel("foraging hours $x$"); plt.ylabel("mass gain $y$ (g)"); plt.legend(); plt.title("The Great Pup Weigh-in")
plt.show()
print("Loss of the guess:", round(loss_ls(0, 3, hours, gain), 2))

In [ ]:
# 💻 The loss is a SURFACE over parameter space (cf. Fig 2.3)
P0, P1 = np.meshgrid(np.linspace(-6, 10, 201), np.linspace(0, 4.5, 201))
Lgrid = ((P0[..., None] + P1[..., None] * hours - gain) ** 2).sum(-1)
fig, ax = plt.subplots(figsize=(6, 4.5))
cs = ax.contourf(P0, P1, np.log(Lgrid), 30, cmap="viridis")
ax.contour(P0, P1, np.log(Lgrid), 15, colors="w", linewidths=0.4)
plt.colorbar(cs, label="log loss"); ax.set_xlabel(r"intercept $\phi_0$"); ax.set_ylabel(r"slope $\phi_1$")
ax.set_title("Loss landscape: one valley, one minimum"); plt.show()

### 🧩 Puzzle 2.1 ⭐ Which way is downhill? (book P2.1)
Differentiate the loss. Write $r_i=\phi_0+\phi_1x_i-y_i$ for the residual of point $i$. Then
$$\frac{\partial L}{\partial \phi_0}=\;?\qquad\frac{\partial L}{\partial \phi_1}=\;?$$
Work it out on paper, then implement `grad_ls`. The checker compares your answer with a finite-difference estimate.

In [ ]:
def grad_ls(phi0, phi1, x, y):
    """Return (dL/dphi0, dL/dphi1) for the least squares loss."""
    # TODO
    raise NotImplementedError

In [ ]:
# 🧪 Check puzzle 2.1
def _t21():
    for p in [(0., 3.), (1.5, 2.2), (-4, 0.7)]:
        g = grad_ls(*p, hours, gain)
        gn = num_grad(lambda q: loss_ls(q[0], q[1], hours, gain), p)
        if not close(g, gn, 1e-3): return False
    return True
check("2.1", 1, _t21, "The gradient of a sum is the sum of the gradients.",
      "Chain rule: d(r^2)/dphi = 2 r * dr/dphi, where dr/dphi0 = 1 and dr/dphi1 = x_i.")

In [ ]:
#@title 🔑 Solution 2.1 — double-click to peek (have a go first!)
peek('2.1')
def grad_ls(phi0, phi1, x, y):
    r = phi0 + phi1 * x - y
    return 2 * np.sum(r), 2 * np.sum(r * x)

In [ ]:
#@title ⚙️ Ready-made helper: `closed_form` (just run me)
def closed_form(x, y):
    xb, yb = x.mean(), y.mean()
    phi1 = np.sum((x - xb) * (y - yb)) / np.sum((x - xb) ** 2)
    return yb - phi1 * xb, phi1
print(closed_form(hours, gain))

In [ ]:
# 💻 Gradient descent: "walk downhill" (Fig 2.4). Watch it zig-zag along a narrow valley.
def gradient_descent_lr(phi, x, y, lr=1e-3, n_steps=4000):
    path = [np.array(phi, float)]
    for _ in range(n_steps):
        g = np.array(grad_ls(*path[-1], x, y))
        path.append(path[-1] - lr * g)
    return np.array(path)

path = gradient_descent_lr([-5, 4.0], hours, gain)
best = closed_form(hours, gain)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].contourf(P0, P1, np.log(Lgrid), 30, cmap="viridis")
ax[0].plot(path[:, 0], path[:, 1], ".-", color=TB["orange"], ms=2, lw=1)
ax[0].plot(*best, "*", color="w", ms=14); ax[0].set_xlabel(r"$\phi_0$"); ax[0].set_ylabel(r"$\phi_1$")
ax[0].set_title("GD path in parameter space")
ax[1].scatter(hours, gain, color=TB["orange"], zorder=3)
for k, c in zip([0, 5, 50, 500, 4000], plt.cm.viridis(np.linspace(0, 0.9, 5))):
    ax[1].plot(xx, f_linear(xx, *path[k]), color=c, label=f"step {k}")
ax[1].legend(); ax[1].set_title("...and the corresponding lines"); plt.show()
print("GD after 4000 steps:", path[-1], " closed form:", np.array(best))

In [ ]:
#@title ⚙️ Ready-made helper: `generative_line` (just run me)
def generative_line(x, y):
    p0, p1 = closed_form(y, x)   # regress x on y
    return -p0 / p1, 1 / p1
a, b = generative_line(hours, gain); d0, d1 = closed_form(hours, gain)
plt.scatter(hours, gain, color=TB["orange"], zorder=3)
plt.plot(xx, d0 + d1 * xx, color=TB["green"], label=f"discriminative  y={d0:.2f}+{d1:.2f}x")
plt.plot(xx, a + b * xx, color=TB["purple"], label=f"generative (inverted)  y={a:.2f}+{b:.2f}x")
plt.legend(); plt.title("Same data, two different lines"); plt.show()
print("The generative fit minimises HORIZONTAL errors, so its inverted line is steeper.\n"
      "The two only agree when the points are perfectly collinear (correlation = ±1).")

---
# Part 3 · Shallow neural networks (Chapter 3): ⛏️ digging burrows with ReLUs
### 📖 Recap
A shallow network with one input, one output and three hidden units has **ten parameters** (eq 3.1):
$$y=\phi_0+\phi_1\,a[\theta_{10}+\theta_{11}x]+\phi_2\,a[\theta_{20}+\theta_{21}x]+\phi_3\,a[\theta_{30}+\theta_{31}x],$$
with the **ReLU** activation $a[z]=\mathrm{ReLU}[z]=\max(0,z)$ (eq 3.2).

It works in three stages:
1. Compute three **linear functions** of the input; these are the *pre-activations*.
2. **Clip** each one at zero with the ReLU to get the hidden units $h_d$ (the *activations*).
3. Take a **weighted sum** of the hidden units and add an offset $\phi_0$.

The result is a **continuous piecewise-linear** function. Each hidden unit contributes one "joint", at $x=-\theta_{d0}/\theta_{d1}$, so $D$ hidden units give at most $D+1$ linear regions. Each region corresponds to one **activation pattern** (which units are active and which are clipped).

**Universal approximation theorem:** with enough hidden units, a shallow net can approximate any continuous function on a compact domain to arbitrary precision.

In [ ]:
# 💻 Build the 10-parameter network. These are the book's own parameters (P3.8 / Fig 3.3).
def relu(z):
    return np.maximum(z, 0.0)

def shallow_1_1_3(x, act, phi, theta):
    """phi = [phi0, phi1, phi2, phi3]; theta = 3x2 array [[th10, th11], [th20, th21], [th30, th31]].
    Returns output y plus the intermediate pre-activations and activations (for plotting)."""
    pre = theta[:, [0]] + theta[:, [1]] * x[None, :]     # (3, N) linear functions
    h = act(pre)                                         # (3, N) hidden units
    y = phi[0] + phi[1:] @ h                             # weighted sum + offset
    return y, pre, h

PHI_BOOK = np.array([-0.23, -1.3, 1.3, 0.66])
THETA_BOOK = np.array([[-0.2, 0.4], [-0.9, 0.9], [1.1, -0.7]])

def plot_decomposition(act, phi, theta, title=""):
    x = np.linspace(0, 2, 400); y, pre, h = shallow_1_1_3(x, act, phi, theta)
    fig, ax = plt.subplots(3, 4, figsize=(12, 6.5), sharex=True)
    for d in range(3):
        ax[d, 0].plot(x, pre[d], color=TB["blue"]); ax[d, 0].set_title(fr"$\theta_{{{d+1}0}}+\theta_{{{d+1}1}}x$")
        ax[d, 1].plot(x, h[d], color=TB["green"]); ax[d, 1].set_title(fr"$h_{d+1}$")
        ax[d, 2].plot(x, phi[d + 1] * h[d], color=TB["orange"]); ax[d, 2].set_title(fr"$\phi_{d+1}h_{d+1}$")
        for c in range(3): ax[d, c].set_ylim(-1, 1)
    for d in range(3): ax[d, 3].axis("off")
    big = fig.add_subplot(1, 4, 4); big.plot(x, y, color="k", lw=2)
    big.set_title(r"$y=\phi_0+\sum_d\phi_dh_d$"); big.set_ylim(-1, 1)
    fig.suptitle(title or "Fig 3.3 recreated", fontweight="bold"); plt.tight_layout(); plt.show()

plot_decomposition(relu, PHI_BOOK, THETA_BOOK)

### 🧩 Puzzle 3.1 ⭐ Where are the joints? What are the slopes? (book P3.3)
- Hidden unit $d$ switches on or off where $\theta_{d0}+\theta_{d1}x=0$, i.e. at $x=-\theta_{d0}/\theta_{d1}$.
- Inside a region, the slope is $\sum_{d\,\in\,\text{active}}\phi_d\,\theta_{d1}$.

Implement both functions. `region_slopes` must return the slopes from **left to right** over the whole real line: four values for three joints.

In [ ]:
def joints(theta):
    """Sorted x-positions of the joints."""
    # TODO
    raise NotImplementedError

def region_slopes(phi, theta):
    """Slopes of the len(theta)+1 linear regions, left to right."""
    # TODO  (tip: pick a test point inside each region and ask which units are active)
    raise NotImplementedError

In [ ]:
# 🧪 Check puzzle 3.1
def _t31():
    j = joints(THETA_BOOK); s = region_slopes(PHI_BOOK, THETA_BOOK)
    x = np.array([j[0] - 1, (j[0] + j[1]) / 2, (j[1] + j[2]) / 2, j[2] + 1]); e = 1e-5
    num = (shallow_1_1_3(x + e, relu, PHI_BOOK, THETA_BOOK)[0] - shallow_1_1_3(x - e, relu, PHI_BOOK, THETA_BOOK)[0]) / (2 * e)
    print("   joints:", np.round(j, 4), " slopes:", np.round(s, 4))
    return close(j, [0.5, 1.0, 1.1 / 0.7], 1e-6) and close(s, num, 1e-5)
check("3.1", 1, _t31, "You can read a ReLU net like a map.",
      "joints = -theta[:,0]/theta[:,1]; a unit is active where theta0 + theta1*x > 0.")

In [ ]:
#@title 🔑 Solution 3.1 — double-click to peek (have a go first!)
peek('3.1')
def joints(theta):
    return np.sort(-theta[:, 0] / theta[:, 1])

def region_slopes(phi, theta):
    j = joints(theta)
    tests = np.concatenate([[j[0] - 1], (j[:-1] + j[1:]) / 2, [j[-1] + 1]])
    slopes = []
    for x in tests:
        active = theta[:, 0] + theta[:, 1] * x > 0
        slopes.append(np.sum(phi[1:][active] * theta[active, 1]))
    return np.array(slopes)
print(joints(THETA_BOOK), region_slopes(PHI_BOOK, THETA_BOOK))

In [ ]:
#@title ⚙️ Ready-made helper: `rescale_unit` (just run me)
def rescale_unit(phi, theta, d, alpha):
    phi, theta = phi.copy(), theta.copy()
    theta[d] *= alpha; phi[d + 1] /= alpha
    return phi, theta

In [ ]:
# 💻 The activation-function zoo (Fig 3.13)
from math import erf
zoo = {
    "ReLU": relu,
    "leaky ReLU (0.1)": lambda z: np.where(z > 0, z, 0.1 * z),
    "sigmoid": lambda z: 1 / (1 + np.exp(-z)),
    "tanh": np.tanh,
    "softplus": lambda z: np.log1p(np.exp(z)),
    "GELU": lambda z: 0.5 * z * (1 + np.vectorize(erf)(z / np.sqrt(2))),
    "Swish (β=1)": lambda z: z / (1 + np.exp(-z)),
    "HardSwish": lambda z: np.where(z < -3, 0, np.where(z > 3, z, z * (z + 3) / 6)),
}
z = np.linspace(-4, 4, 400)
fig, axs = plt.subplots(2, 4, figsize=(12, 5), sharex=True)
for ax, (name, fn) in zip(axs.ravel(), zoo.items()):
    ax.plot(z, fn(z), color=TB["green"], lw=2); ax.set_title(name); ax.axhline(0, color="k", lw=0.5)
plt.tight_layout(); plt.show()

In [ ]:
def vigilance(t):
    return 0.6 + 0.3 * np.sin(2 * np.pi * t) * np.exp(-1.5 * t) + 0.15 * np.cos(9 * t)

def shallow_1d(x, phi0, phis, theta0s, theta1s):
    """General 1-input/1-output shallow ReLU net with D units."""
    return phi0 + relu(theta0s[:, None] + theta1s[:, None] * x[None, :]).T @ phis

In [ ]:
#@title ⚙️ Ready-made helper: `make_interpolating_net` (just run me)
def make_interpolating_net(tk, vk):
    s = np.diff(vk) / np.diff(tk)
    phis = np.diff(np.concatenate([[0.0], s]))
    return vk[0], phis, -tk[:-1], np.ones(len(tk) - 1)

In [ ]:
# 💻 More units -> more regions -> smaller error
t = np.linspace(0, 1, 1000)
fig, axs = plt.subplots(1, 4, figsize=(13, 3), sharey=True)
for ax, D in zip(axs, [3, 6, 12, 48]):
    tk = np.linspace(0, 1, D + 1); y = shallow_1d(t, *make_interpolating_net(tk, vigilance(tk)))
    ax.plot(t, vigilance(t), "--", color=TB["grey"]); ax.plot(t, y, color=TB["green"], lw=2)
    ax.set_title(f"D={D}  max err={np.max(np.abs(y - vigilance(t))):.3f}")
axs[0].set_ylabel("vigilance"); plt.tight_layout(); plt.show()

### 📖 Multivariate inputs: regions become convex polygons
With $D_i=2$ inputs, each hidden unit's joint becomes a **line** across the input plane, and the regions become **convex polygons** (Fig 3.8). **Zaslavsky (1975):** $D$ hyperplanes cut $\mathbb{R}^{D_i}$ into at most
$$\sum_{j=0}^{D_i}\binom{D}{j}$$
regions. Below we colour each pixel of the plane by its activation pattern and count the distinct colours.

In [ ]:
def count_regions_2d(Om, b, lim=4, n=600):
    g = np.linspace(-lim, lim, n); X1, X2 = np.meshgrid(g, g)
    pre = b[:, None] + Om @ np.stack([X1.ravel(), X2.ravel()])
    codes = (pre > 0).astype(int).T @ (2 ** np.arange(len(b)))
    return codes.reshape(n, n), len(np.unique(codes))

r = np.random.default_rng(5)
fig, axs = plt.subplots(1, 3, figsize=(12, 4))
for ax, D in zip(axs, [3, 5, 12]):
    img, n = count_regions_2d(r.normal(size=(D, 2)), r.normal(size=D))
    ax.imshow(img % 17, cmap="tab20", origin="lower", extent=[-4, 4, -4, 4])
    ax.set_title(f"D={D} hidden units → {n} regions"); ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
plt.tight_layout(); plt.show()

In [ ]:
#@title ⚙️ Ready-made helper: `max_regions`, `n_params_shallow` (just run me)
def max_regions(D, Di):
    return sum(math.comb(D, j) for j in range(min(Di, D) + 1))

def n_params_shallow(Di, D, Do):
    return D * (Di + 1) + Do * (D + 1)

### 🏆 Challenge 3 ⭐⭐ Dig the burrow
The mob's main burrow has a known cross-section (the dashed line below). Hand-pick the ten parameters of a **3-unit** shallow net so that it matches the burrow profile on $x\in[0,1]$.
- 🥉 MSE < 0.01
- 🥈 MSE < 0.001
- 🥇 exact (MSE < 1e-10)

*Strategy: find the three joints and the slope changes at each joint, then use units of the form $\phi_d\,\mathrm{ReLU}[x-c_d]$.*

In [ ]:
def burrow(x):
    return np.select([x < 0.2, x < 0.5, x < 0.7], [0.0, -2 * (x - 0.2), -0.6 + 3 * (x - 0.5)], 0.0)
xb = np.linspace(0, 1, 501)
plt.figure(figsize=(6, 2.6)); plt.fill_between(xb, burrow(xb), -0.8, color=TB["sand"])
plt.plot(xb, burrow(xb), "k--"); plt.title("Burrow cross-section (sand in beige)"); plt.ylim(-0.8, 0.2); plt.show()

In [ ]:
my_phi = np.array([0.0, 0.0, 0.0, 0.0])                  # TODO [phi0, phi1, phi2, phi3]
my_theta = np.array([[0.0, 1.0], [0.0, 1.0], [0.0, 1.0]])  # TODO [[th10, th11], ...]

In [ ]:
# 🧪 Check puzzle C3
def _tc3():
    mse = np.mean((shallow_1_1_3(xb, relu, my_phi, my_theta)[0] - burrow(xb)) ** 2)
    print(f"   MSE = {mse:.2e}  →  {medal(mse, 1e-2, 1e-3, 1e-10)}")
    plt.figure(figsize=(6, 2.6)); plt.plot(xb, burrow(xb), "k--", label="burrow")
    plt.plot(xb, shallow_1_1_3(xb, relu, my_phi, my_theta)[0], color=TB["green"], lw=2, label="your net")
    plt.legend(); plt.show()
    return mse < 1e-10
check("C3", 2, _tc3, "Perfect burrow. The mob moves in tonight.",
      "Joints at 0.2, 0.5, 0.7. Slope goes 0 → -2 → +3 → 0, so the slope changes are -2, +5, -3.")

In [ ]:
#@title 🔑 Solution C3 — double-click to peek (have a go first!)
peek('C3')
my_phi = np.array([0.0, -2.0, 5.0, -3.0])
my_theta = np.array([[-0.2, 1.0], [-0.5, 1.0], [-0.7, 1.0]])

---
# Part 4 · Deep neural networks (Chapter 4): 🏜️ folding the dune sea
### 📖 Recap
**Composing** two shallow networks, $x\to y\to y'$, gives a deep network. The first network **folds** the input space onto itself, so several $x$ values map to the same $y$. The second network then applies its function, and that function is **replicated** at every fold (Figs 4.1 and 4.3).

Substituting the first network into the second gives a two-layer network (eqs 4.7–4.9):
$$\mathbf{h}=a[\boldsymbol\theta_0+\boldsymbol\theta x],\quad \mathbf{h}'=a[\boldsymbol\psi_0+\boldsymbol\Psi\mathbf{h}],\quad y'=\phi_0'+\boldsymbol\phi'\mathbf{h}' .$$
In the composed case $\boldsymbol\Psi$ is forced to be an outer product, so a general deep network is *more* expressive than the composition.

**General $K$-layer network** (eq 4.15):
$$\mathbf{h}_1=a[\boldsymbol\beta_0+\boldsymbol\Omega_0\mathbf{x}],\;\; \mathbf{h}_{k+1}=a[\boldsymbol\beta_k+\boldsymbol\Omega_k\mathbf{h}_k],\;\; \mathbf{y}=\boldsymbol\beta_K+\boldsymbol\Omega_K\mathbf{h}_K,$$
with $\boldsymbol\Omega_k\in\mathbb{R}^{D_{k+1}\times D_k}$ and $\boldsymbol\beta_k\in\mathbb{R}^{D_{k+1}}$.

**Shallow vs deep** (1 input, 1 output, $D>2$ units per layer):

| | parameters | max linear regions |
|---|---|---|
| shallow | $3D+1$ | $D+1$ |
| $K$ layers | $3D+1+(K-1)D(D+1)$ | $(D+1)^K$ |

In [ ]:
# 💻 Composition = folding.  Net 1 zig-zags [-1,1] -> [-1,1] with three alternating-slope pieces.
def net1(x):
    return shallow_1d(x, -1.0, np.array([3.0, -6.0, 6.0]), np.array([1.0, 1/3, -1/3]), np.ones(3))
def net2(y):   # joints at -0.3 and 0.5 (third joint at -2.0 is outside [-1, 1])
    return shallow_1d(y, -0.5, np.array([2.0, -3.0, 1.0]), np.array([-0.3, -0.5, 2.0]), np.array([-1.0, 1.0, 1.0]))

def count_regions_1d(fn, lo=-1, hi=1, n=400_001, tol=1e-6):
    """Count linear regions of a piecewise-linear fn on [lo, hi] by finding slope changes."""
    x = np.linspace(lo, hi, n); s = np.diff(fn(x)) / np.diff(x)
    flags = np.abs(np.diff(s)) > tol * (1 + np.abs(s).max())
    starts = np.flatnonzero(flags & ~np.concatenate([[False], flags[:-1]]))   # merge neighbours
    return len(starts) + 1

x = np.linspace(-1, 1, 2001)
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
ax[0].plot(x, net1(x), color=TB["blue"], lw=2); ax[0].axhline(0.4, color=TB["orange"], ls=":")
ax[0].set_title(f"net 1 (folds): {count_regions_1d(net1)} regions"); ax[0].set_xlabel("x"); ax[0].set_ylabel("y")
ax[1].plot(x, net2(x), color=TB["purple"], lw=2); ax[1].set_title(f"net 2: {count_regions_1d(net2)} regions on [-1,1]")
ax[1].set_xlabel("y"); ax[1].set_ylabel("y'")
ax[2].text(0.5, 0.5, "?", fontsize=60, ha="center", va="center", color=TB["grey"], transform=ax[2].transAxes)
ax[2].set_title("net2(net1(x)) = ?"); plt.tight_layout(); plt.show()
print("(the dotted line in panel 1: three different x's land on the same y, so net 2 sees them as one point)")

In [ ]:
# 🧩 Puzzle 4.1 ⭐  Predict BEFORE you run the next line:
#    how many linear regions does net2(net1(x)) have on x in [-1, 1]?
my_prediction = None  # TODO

In [ ]:
# 🧪 Check puzzle 4.1
def _t41():
    if my_prediction is None: raise NotImplementedError
    truth = count_regions_1d(lambda x: net2(net1(x)))
    plt.figure(figsize=(4.5, 3.2)); plt.plot(x, net2(net1(x)), color=TB["green"], lw=2)
    plt.title(f"composition: {truth} regions"); plt.xlabel("x"); plt.ylabel("y'"); plt.show()
    return my_prediction == truth
check("4.1", 1, _t41, "3 folds × 3 regions = 9. Net 2's function gets copied (and flipped) onto each fold.",
      "Net 1 maps each of its 3 pieces onto the WHOLE of [-1, 1].")

In [ ]:
#@title 🔑 Solution 4.1 — double-click to peek (have a go first!)
peek('4.1')
my_prediction = 9

In [ ]:
# 💻 Random initialisation for any layer sizes (we'll revisit the variance choice in Part 7)
def init_params(dims, sigma2=None, rng=np.random.default_rng(0)):
    """dims = [Di, D1, ..., DK, Do]. Normal weights (He variance 2/D_in by default), zero biases."""
    betas, omegas = [], []
    for d_in, d_out in zip(dims[:-1], dims[1:]):
        s2 = 2.0 / d_in if sigma2 is None else sigma2
        omegas.append(rng.normal(0, np.sqrt(s2), (d_out, d_in))); betas.append(np.zeros((d_out, 1)))
    return betas, omegas

In [ ]:
#@title ⚙️ Ready-made helper: `deep_forward` (just run me)
def deep_forward(x, betas, omegas, act=relu):
    all_f, all_h = [], [x]
    K = len(omegas) - 1
    for k in range(K + 1):
        f = betas[k] + omegas[k] @ all_h[-1]
        all_f.append(f)
        if k < K:
            all_h.append(act(f))
    return all_f[-1], all_f, all_h

In [ ]:
#@title ⚙️ Ready-made helper: `n_params_deep_1d`, `max_regions_deep_1d` (just run me)
def n_params_deep_1d(D, K):
    return 3 * D + 1 + (K - 1) * D * (D + 1)
def max_regions_deep_1d(D, K):
    return (D + 1) ** K

In [ ]:
plt.figure(figsize=(6, 4))
for K in range(1, 6):
    Ds = np.arange(3, 60)
    plt.loglog([n_params_deep_1d(D, K) for D in Ds], [max_regions_deep_1d(D, K) for D in Ds], label=f"K={K}")
plt.plot(471, 161051, "o", color="k", ms=8, mfc="none"); plt.annotate("K=5, D=10", (471, 161051), xytext=(600, 3e3))
plt.xlabel("number of parameters"); plt.ylabel("max linear regions"); plt.legend(); plt.title("Fig 4.7a")
plt.show()

In [ ]:
# 💻 Reality check: RANDOM nets come nowhere near the maximum region count.
rr = np.random.default_rng(3); res = {}
for K, D in [(1, 30), (2, 6), (4, 3)]:
    cnts = []
    for _ in range(20):
        b, o = init_params([1] + [D] * K + [1], rng=rr)
        b = [bi + rr.normal(0, 0.5, bi.shape) for bi in b]
        cnts.append(count_regions_1d(lambda x: deep_forward(x[None, :], b, o)[0][0], -3, 3, n=100_001))
    res[(K, D)] = (n_params_deep_1d(D, K), np.mean(cnts), max_regions_deep_1d(D, K))
for (K, D), (p, m, mx) in res.items():
    print(f"K={K}, D={D:2d}: {p:3d} params | random init: {m:5.1f} regions on [-3,3] | theoretical max {mx}")
print("\nThe book's caveat (§4.5.2): many regions are possible, but they come with complex dependencies and symmetries.")

---
# Part 5 · Loss functions (Chapter 5): 🔭 sentinels, alarm calls, scorpions and lost burrows
### 📖 The recipe (§5.2)
Instead of predicting $\mathbf{y}$ directly, the network predicts the **parameters $\boldsymbol\theta$ of a distribution** over $\mathbf{y}$.
1. Choose a distribution $Pr(\mathbf{y}\,|\,\boldsymbol\theta)$ defined on the output domain.
2. Let the network predict the parameters: $\boldsymbol\theta=\mathbf{f}[\mathbf{x},\boldsymbol\phi]$.
3. Train by **maximum likelihood**. Because $\log$ is monotonic and we prefer minimizing, minimize the **negative log-likelihood**:
$$\hat{\boldsymbol\phi}=\underset{\boldsymbol\phi}{\mathrm{argmin}}\Big[-\sum_{i=1}^{I}\log Pr\big(\mathbf{y}_i\,|\,\mathbf{f}[\mathbf{x}_i,\boldsymbol\phi]\big)\Big].$$
4. **Inference**: return the full distribution, or its maximum $\hat{\mathbf{y}}=\mathrm{argmax}_{\mathbf{y}}\,Pr(\mathbf{y}\,|\,\mathbf{f}[\mathbf{x},\hat{\boldsymbol\phi}])$.

| output domain | distribution | final activation | loss |
|---|---|---|---|
| $y\in\mathbb{R}$ | normal | none | least squares |
| $y\in\{0,1\}$ | Bernoulli | logistic sigmoid | binary cross-entropy |
| $y\in\{1..K\}$ | categorical | softmax | multiclass cross-entropy |
| $y\in\{0,1,2,\dots\}$ | Poisson | $\exp$ | Poisson NLL |
| $y\in(-\pi,\pi]$ | von Mises | none | $-\kappa\cos(y-\mu)$ |

## 5A · Univariate regression: least squares is a *derived* result
With $Pr(y|\mu,\sigma^2)=\frac{1}{\sqrt{2\pi\sigma^2}}\exp\!\big[-\frac{(y-\mu)^2}{2\sigma^2}\big]$ and $\mu=\mathrm{f}[x,\boldsymbol\phi]$:
$$L[\boldsymbol\phi]=-\sum_i\log Pr(y_i|\mathrm{f}[x_i,\boldsymbol\phi],\sigma^2)=\frac{I}{2}\log(2\pi\sigma^2)+\sum_i\frac{(y_i-\mathrm{f}[x_i,\boldsymbol\phi])^2}{2\sigma^2}.$$
The first term doesn't depend on $\boldsymbol\phi$, and $2\sigma^2$ is a positive constant, so the argmin is exactly the **least squares** solution (eq 5.11). The Gaussian assumption is what justifies the loss we used in Part 2.

In [ ]:
#@title ⚙️ Ready-made helper: `normal_nll`, `sigma2_mle` (just run me)
def normal_nll(y, mu, sigma2):
    return np.sum(0.5 * np.log(2 * np.pi * sigma2) + (y - mu) ** 2 / (2 * sigma2))
def sigma2_mle(y, mu):
    return np.mean((y - mu) ** 2)

### 📖 Heteroscedastic regression (§5.3.4): pups get *more variable* as they grow
Give the network **two outputs**: a mean $\mu=\mathrm{f}_1[x,\boldsymbol\phi]$ and a variance $\sigma^2=\mathrm{softplus}[\mathrm{f}_2[x,\boldsymbol\phi]]$, which keeps the variance positive. Minimize
$$L=\sum_i\Big[\tfrac12\log(2\pi\sigma_i^2)+\frac{(y_i-\mu_i)^2}{2\sigma_i^2}\Big].$$

In [ ]:
#@title ⚙️ Ready-made helper: `hetero_nll` (just run me)
def hetero_nll(y, out):
    mu = out[:, 0]; s2 = torch.nn.functional.softplus(out[:, 1]) + 1e-6
    return (0.5 * torch.log(2 * math.pi * s2) + (y - mu) ** 2 / (2 * s2)).mean()

In [ ]:
# 💻 Train the heteroscedastic pup-growth model
if HAS_TORCH:
    torch.manual_seed(0); rr = np.random.default_rng(4)
    age = rr.uniform(0, 1, 400)                                     # age (scaled 0..1 ≈ 0..120 days)
    mass = 30 + 220 * (1 - np.exp(-2.5 * age)) + rr.normal(0, 4 + 30 * age)   # noise grows with age
    X = torch.tensor(age, dtype=torch.float32)[:, None]; Y = torch.tensor((mass - 150) / 50, dtype=torch.float32)
    net = torch.nn.Sequential(torch.nn.Linear(1, 32), torch.nn.ReLU(), torch.nn.Linear(32, 32),
                              torch.nn.ReLU(), torch.nn.Linear(32, 2))
    opt = torch.optim.Adam(net.parameters(), lr=1e-2)
    for step in range(2000):
        opt.zero_grad(); loss = hetero_nll(Y, net(X)); loss.backward(); opt.step()
    with torch.no_grad():
        g = torch.linspace(0, 1, 200)[:, None]; o = net(g)
        mu = o[:, 0].numpy() * 50 + 150; sd = np.sqrt(torch.nn.functional.softplus(o[:, 1]).numpy() + 1e-6) * 50
    plt.scatter(age * 120, mass, s=6, color=TB["orange"], alpha=0.6, label="pups")
    plt.plot(g[:, 0] * 120, mu, color=TB["green"], lw=2, label=r"$\mu$")
    plt.fill_between(g[:, 0].numpy() * 120, mu - 2 * sd, mu + 2 * sd, color=TB["lgreen"], alpha=0.3, label=r"$\mu\pm2\sigma$")
    plt.xlabel("age (days)"); plt.ylabel("mass (g)"); plt.legend(); plt.title("Heteroscedastic regression"); plt.show()

## 5B · Binary classification: 🔭 will this meerkat go on sentinel duty?
Meerkats tend to stand guard once they've **eaten well**: a full stomach makes guarding cheap. Our feature is $x$ = grams of food eaten in the last hour, and the label is $y=1$ if the animal goes on guard.

- **Bernoulli**: $Pr(y|\lambda)=(1-\lambda)^{1-y}\lambda^{y}$, with $\lambda=\mathrm{sig}[\mathrm{f}[x,\boldsymbol\phi]]$ and $\mathrm{sig}[z]=\frac{1}{1+e^{-z}}$.
- **Binary cross-entropy** (eq 5.20): $L=\sum_i -(1-y_i)\log\big(1-\mathrm{sig}[\mathrm{f}_i]\big)-y_i\log\mathrm{sig}[\mathrm{f}_i]$.

### 🧩 Puzzle 5.3 ⭐⭐ The prettiest derivative in deep learning
Show that $\dfrac{\partial \ell_i}{\partial \mathrm{f}_i}=\mathrm{sig}[\mathrm{f}_i]-y_i$. (Use $\mathrm{sig}'[z]=\mathrm{sig}[z](1-\mathrm{sig}[z])$.) Then implement both functions below. The loss must be **numerically stable**: no `log(0)` for large $|f|$. Hint: $-\log\mathrm{sig}[f]=\mathrm{softplus}[-f]$, and `np.logaddexp(0, z)` computes $\log(1+e^{z})$ safely.

In [ ]:
def bce_from_logits(f, y):
    """Sum of binary cross-entropy terms, computed stably from logits f."""
    # TODO
    raise NotImplementedError

def bce_grad_logits(f, y):
    """d(loss)/df, elementwise."""
    # TODO
    raise NotImplementedError

In [ ]:
# 🧪 Check puzzle 5.3
def _t53():
    f = np.array([-3.0, 0.2, 4.0, 40.0, -40.0]); y = np.array([0, 1, 1, 0, 1.0])
    sg = 1 / (1 + np.exp(-f[:3]))
    ref3 = np.sum(-(1 - y[:3]) * np.log(1 - sg) - y[:3] * np.log(sg))
    stable = np.isfinite(bce_from_logits(f, y)) and np.isclose(bce_from_logits(f[3:], y[3:]), 80, atol=1e-6)
    g_num = num_grad(lambda q: bce_from_logits(q, y), f)
    return np.isclose(bce_from_logits(f[:3], y[:3]), ref3) and stable and close(bce_grad_logits(f, y), g_num, 1e-5)
check("5.3", 2, _t53, "sig(f) - y: the error signal is just 'prediction minus truth'.",
      "loss = sum(y*logaddexp(0,-f) + (1-y)*logaddexp(0,f)); grad = 1/(1+exp(-f)) - y")

In [ ]:
#@title 🔑 Solution 5.3 — double-click to peek (have a go first!)
peek('5.3')
def bce_from_logits(f, y):
    return np.sum(y * np.logaddexp(0, -f) + (1 - y) * np.logaddexp(0, f))
def bce_grad_logits(f, y):
    return 1 / (1 + np.exp(-f)) - y

In [ ]:
# 💻 Plot the loss vs predicted probability (book P5.2), then fit logistic regression from scratch
lam = np.linspace(1e-3, 1 - 1e-3, 300)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(lam, -np.log(1 - lam), label="y = 0", color=TB["blue"]); ax[0].plot(lam, -np.log(lam), label="y = 1", color=TB["orange"])
ax[0].set_xlabel(r"$\mathrm{sig}[f]$"); ax[0].set_ylabel("loss"); ax[0].legend(); ax[0].set_title("Confidently wrong = very expensive")

rr = np.random.default_rng(7); food = rr.uniform(0, 30, 120)
guard = (rr.uniform(size=120) < 1 / (1 + np.exp(-(food - 15) / 3))).astype(float)
phi = np.zeros(2)
for _ in range(5000):
    fz = phi[0] + phi[1] * (food - 15) / 10
    gz = bce_grad_logits(fz, guard)
    phi -= 0.01 * np.array([gz.sum(), (gz * (food - 15) / 10).sum()])
xs = np.linspace(0, 30, 200)
ax[1].scatter(food, guard + rr.normal(0, 0.02, 120), s=10, color=TB["grey"], label="observed")
ax[1].plot(xs, 1 / (1 + np.exp(-(phi[0] + phi[1] * (xs - 15) / 10))), color=TB["green"], lw=2, label="Pr(guard | food)")
ax[1].set_xlabel("food eaten in last hour (g)"); ax[1].legend(); ax[1].set_title("Selfish sentinels"); plt.tight_layout(); plt.show()

## 5C · Multiclass classification: 🦅🐺🐍 decoding alarm calls
Meerkats give **different alarm calls for different predators**. Two acoustic features (peak frequency and call duration) should let us classify the call as **eagle / jackal / cobra**.

- **Categorical** distribution with $Pr(y=k|\mathbf{x})=\mathrm{softmax}_k[\mathbf{f}[\mathbf{x},\boldsymbol\phi]]=\dfrac{\exp[\mathrm{f}_k]}{\sum_{k'}\exp[\mathrm{f}_{k'}]}$ (eq 5.22).
- **Multiclass cross-entropy** (eq 5.24): $L=-\sum_i\Big(\mathrm{f}_{y_i}[\mathbf{x}_i,\boldsymbol\phi]-\log\sum_k\exp\big[\mathrm{f}_k[\mathbf{x}_i,\boldsymbol\phi]\big]\Big)$.
- Its gradient with respect to the logits is $\mathrm{softmax}[\mathbf{f}]-\mathbf{e}_{y}$, where $\mathbf{e}_y$ is the one-hot label. It's the same "prediction minus truth" as in 5B.

### 🧩 Puzzle 5.4 ⭐ A softmax that never overflows
`np.exp(1000)` is `inf`. Use the fact that $\mathrm{softmax}[\mathbf{z}]=\mathrm{softmax}[\mathbf{z}-c]$ for any constant $c$. Columns are examples, so normalise over **axis 0**.

In [ ]:
def softmax(z):
    """z: (K, N) logits -> (K, N) probabilities, normalised over axis 0, stable for huge z."""
    # TODO
    raise NotImplementedError

In [ ]:
# 🧪 Check puzzle 5.4
def _t54():
    z = np.array([[1000., 1.], [1001., 2.], [1002., 3.]])
    p = softmax(z); ref = np.exp(z[:, 1]) / np.exp(z[:, 1]).sum()
    return np.all(np.isfinite(p)) and close(p[:, 0], ref) and close(p[:, 1], ref) and close(p.sum(0), [1, 1])
check("5.4", 1, _t54, "Subtracting the max: the oldest trick in numerical deep learning.",
      "z = z - z.max(axis=0, keepdims=True)")

In [ ]:
#@title 🔑 Solution 5.4 — double-click to peek (have a go first!)
peek('5.4')
def softmax(z):
    e = np.exp(z - z.max(axis=0, keepdims=True))
    return e / e.sum(axis=0, keepdims=True)

In [ ]:
# 💻 Linear softmax classifier from scratch: y = softmax(beta + Omega x), trained by gradient descent
rr = np.random.default_rng(11)
centres = np.array([[4.2, 0.20], [2.0, 0.55], [3.1, 1.10]])      # (kHz, s)  eagle / jackal / cobra
names = ["eagle", "jackal", "cobra"]; cols = [TB["blue"], TB["orange"], TB["green"]]
Xc = np.concatenate([rr.normal(c, [0.45, 0.15], (80, 2)) for c in centres]).T          # (2, 240)
yc = np.repeat([0, 1, 2], 80); mu_, sd_ = Xc.mean(1, keepdims=True), Xc.std(1, keepdims=True)
Xn = (Xc - mu_) / sd_; onehot = np.eye(3)[yc].T
Om, be = np.zeros((3, 2)), np.zeros((3, 1)); losses = []
for it in range(600):
    P = softmax(be + Om @ Xn); G = P - onehot                         # dL/df  (3, N)
    Om -= 0.5 * G @ Xn.T / Xn.shape[1]; be -= 0.5 * G.mean(1, keepdims=True)
    losses.append(-np.mean(np.log(P[yc, np.arange(len(yc))])))
g1, g2 = np.meshgrid(np.linspace(0.5, 5.8, 300), np.linspace(-0.3, 1.8, 300))
G = (np.stack([g1.ravel(), g2.ravel()]) - mu_) / sd_; pred = softmax(be + Om @ G).argmax(0).reshape(g1.shape)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].contourf(g1, g2, pred, levels=[-.5, .5, 1.5, 2.5], colors=cols, alpha=0.18)
for k in range(3): ax[0].scatter(*Xc[:, yc == k], s=10, color=cols[k], label=names[k])
ax[0].set_xlabel("peak frequency (kHz)"); ax[0].set_ylabel("duration (s)"); ax[0].legend(); ax[0].set_title("Alarm-call decoder")
ax[1].plot(losses, color=TB["green"]); ax[1].set_xlabel("iteration"); ax[1].set_ylabel("mean cross-entropy"); plt.show()
print("training accuracy:", np.mean(softmax(be + Om @ Xn).argmax(0) == yc))

## 5D · Counts: 🦂 scorpions per hour (book P5.6)
Meerkats eat scorpions (they're largely immune to the venom). The number caught per hour, $y\in\{0,1,2,\dots\}$, rises with temperature. Use a **Poisson** distribution $Pr(y=k)=\frac{\lambda^ke^{-\lambda}}{k!}$, with the rate predicted through an $\exp$ so that it stays positive: $\lambda=\exp[\mathrm{f}[x,\boldsymbol\phi]]$.

In [ ]:
#@title ⚙️ Ready-made helper: `poisson_nll`, `poisson_grad_f`, `fit_poisson` (just run me)
def poisson_nll(y, f):
    return np.sum(np.exp(f) - y * f + np.array([math.lgamma(k + 1) for k in np.atleast_1d(y)]))
def poisson_grad_f(y, f):
    return np.exp(f) - y
def fit_poisson(x, y, lr=1e-3, n_steps=5000):
    phi = np.zeros(2)
    for _ in range(n_steps):
        g = poisson_grad_f(y, phi[0] + phi[1] * x)
        phi -= lr * np.array([g.sum(), (g * x).sum()])
    return phi

## 5E · Directions: 🧭 which way is the burrow? (book P5.3)
At dusk each foraging group heads home. The heading $y\in(-\pi,\pi]$ is an **angle**, and angles wrap around: $+179°$ and $-179°$ are only 2° apart. The **von Mises** distribution handles this:
$$Pr(y|\mu,\kappa)=\frac{\exp[\kappa\cos(y-\mu)]}{2\pi\,\mathrm{Bessel}_0[\kappa]}\quad\Rightarrow\quad L=-\sum_i\kappa\cos(y_i-\mu_i)+\text{const}.$$

### 🧩 Puzzle 5.6 ⭐⭐⭐ The naive average sends the mob the wrong way
For a constant model $\mu$, set $\partial L/\partial\mu=0$ and show that $\sum_i\sin(y_i-\mu)=0$, which gives $\hat\mu=\mathrm{atan2}\big(\sum_i\sin y_i,\ \sum_i\cos y_i\big)$. Implement it, and compare with `np.mean` on headings that cluster around due **west** ($\pm\pi$).

In [ ]:
def vonmises_mu_mle(angles):
    # TODO
    raise NotImplementedError

In [ ]:
# 🧪 Check puzzle 5.6
def _t56():
    rr = np.random.default_rng(3)
    heads = np.angle(np.exp(1j * (np.pi + rr.normal(0, 0.3, 50))))      # around ±pi, wrapped
    mu = vonmises_mu_mle(heads); naive = heads.mean()
    print(f"   naive mean = {np.degrees(naive):7.1f}°   von Mises MLE = {np.degrees(mu):7.1f}°   (truth: ±180°)")
    grid = np.linspace(-np.pi, np.pi, 100001); Lg = [-np.sum(np.cos(heads - m)) for m in grid[::100]]
    m_best = grid[::100][np.argmin(Lg)]
    return abs(np.angle(np.exp(1j * (mu - m_best)))) < 0.01
check("5.6", 3, _t56, "Least squares on angles would send the mob off due EAST.",
      "np.arctan2(np.sin(a).sum(), np.cos(a).sum())")

In [ ]:
#@title 🔑 Solution 5.6 — double-click to peek (have a go first!)
peek('5.6')
def vonmises_mu_mle(angles):
    return np.arctan2(np.sin(angles).sum(), np.cos(angles).sum())

### 📖 Cross-entropy, KL divergence and why it's all the same thing (§5.7)
Let $q(y)$ be the empirical distribution of the data, a set of spikes at the $y_i$. Minimizing the KL divergence $D_{KL}[q\,\|\,Pr(y|\boldsymbol\theta)]=\int q\log q\,dy-\int q\log Pr(y|\boldsymbol\theta)\,dy$ over $\boldsymbol\theta$ drops the first term, since it doesn't depend on $\boldsymbol\theta$. What's left is the **cross-entropy** $-\frac1I\sum_i\log Pr(y_i|\boldsymbol\theta)$: our negative log-likelihood. So "cross-entropy loss" and "maximum likelihood" are two names for the same objective.

---
# Part 6 · Fitting models (Chapter 6): 🏃 race to the burrow
### 📖 Recap
**Gradient descent** (eq 6.3): $\boldsymbol\phi\leftarrow\boldsymbol\phi-\alpha\,\dfrac{\partial L}{\partial\boldsymbol\phi}$, where $\alpha$ is the learning rate.

Linear-regression losses are **convex**: every chord lies above the surface, so there's one minimum. Neural network losses are **non-convex**, with local minima and saddle points. The book explores this with the 2-parameter **Gabor model** (eq 6.8):
$$\mathrm{f}[x,\boldsymbol\phi]=\sin[\phi_0+0.06\,\phi_1x]\cdot\exp\!\Big(-\frac{(\phi_0+0.06\,\phi_1x)^2}{32}\Big).$$

**SGD** (eq 6.10) takes each step using a random **minibatch** $\mathcal{B}_t$: $\;\boldsymbol\phi_{t+1}\leftarrow\boldsymbol\phi_t-\alpha\sum_{i\in\mathcal{B}_t}\partial\ell_i/\partial\boldsymbol\phi$. One pass through the whole dataset is an **epoch**.

**Momentum** (eq 6.11): $\;\mathbf{m}_{t+1}\leftarrow\beta\mathbf{m}_t+(1-\beta)\sum_{i\in\mathcal{B}_t}\partial\ell_i/\partial\boldsymbol\phi,\quad \boldsymbol\phi_{t+1}\leftarrow\boldsymbol\phi_t-\alpha\,\mathbf{m}_{t+1}$.
**Nesterov** (eq 6.12) evaluates the gradient at the look-ahead point $\boldsymbol\phi_t-\alpha\beta\,\mathbf{m}_t$ instead.

**Adam** (eqs 6.15–6.17):
$$\mathbf{m}_{t+1}\leftarrow\beta\mathbf{m}_t+(1-\beta)\mathbf{g}_t,\quad \mathbf{v}_{t+1}\leftarrow\gamma\mathbf{v}_t+(1-\gamma)\mathbf{g}_t^2,$$
$$\tilde{\mathbf{m}}_{t+1}=\frac{\mathbf{m}_{t+1}}{1-\beta^{t+1}},\quad\tilde{\mathbf{v}}_{t+1}=\frac{\mathbf{v}_{t+1}}{1-\gamma^{t+1}},\quad \boldsymbol\phi_{t+1}\leftarrow\boldsymbol\phi_t-\alpha\frac{\tilde{\mathbf{m}}_{t+1}}{\sqrt{\tilde{\mathbf{v}}_{t+1}}+\epsilon}.$$
The bias correction fixes the too-small early estimates (since $\mathbf{m}_0=\mathbf{v}_0=\mathbf{0}$). Dividing by $\sqrt{\tilde{\mathbf v}}$ normalises the step size in every direction.

In [ ]:
# 💻 The Gabor model and its dataset (28 points, true phi = [0, 16.6], as in Fig 6.3)
def gabor(x, phi):
    u = phi[0] + 0.06 * phi[1] * x
    return np.sin(u) * np.exp(-u ** 2 / 32.0)

rr = np.random.default_rng(8)
xg = np.sort(rr.uniform(-15, 15, 28)); yg = gabor(xg, [0.0, 16.6]) + rr.normal(0, 0.1, 28)
def gabor_loss(phi, x=xg, y=yg):
    return np.sum((gabor(x, phi) - y) ** 2)

A0, A1 = np.meshgrid(np.linspace(-10, 10, 241), np.linspace(2.5, 22.5, 241))
LG = np.array([[gabor_loss([a, b]) for a, b in zip(r0, r1)] for r0, r1 in zip(A0, A1)])
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(xg, yg, color=TB["orange"]); xx = np.linspace(-15, 15, 400)
ax[0].plot(xx, gabor(xx, [0, 16.6]), color=TB["green"]); ax[0].set_title("Gabor data")
c = ax[1].contourf(A0, A1, LG, 30, cmap="viridis"); plt.colorbar(c, ax=ax[1])
ax[1].set_xlabel(r"$\phi_0$"); ax[1].set_ylabel(r"$\phi_1$"); ax[1].set_title("Non-convex loss: many valleys"); plt.show()

In [ ]:
#@title ⚙️ Ready-made helper: `gabor_grad` (just run me)
def gabor_grad(phi, x=xg, y=yg):
    u = phi[0] + 0.06 * phi[1] * x; e = np.exp(-u ** 2 / 32.0)
    dg = np.cos(u) * e - (u / 16.0) * np.sin(u) * e
    r = 2 * (np.sin(u) * e - y)
    return np.array([np.sum(r * dg), np.sum(r * dg * 0.06 * x)])

In [ ]:
# 💻 Full-batch GD vs SGD from the same starting points (Fig 6.5)
def gd(grad_fn, phi, lr, n):
    path = [np.array(phi, float)]
    for _ in range(n): path.append(path[-1] - lr * grad_fn(path[-1]))
    return np.array(path)

def sgd_gabor(phi, lr, n, batch=7, seed=0):
    r = np.random.default_rng(seed); path = [np.array(phi, float)]; order = r.permutation(28); pos = 0
    for t in range(n):
        lr_t = lr * (0.5 ** (t // 600))                        # learning-rate schedule: halve every 600 steps
        if pos + batch > 28: order, pos = r.permutation(28), 0  # new epoch
        idx = order[pos:pos + batch]; pos += batch
        path.append(path[-1] - lr_t * gabor_grad(path[-1], xg[idx], yg[idx]))
    return np.array(path)

starts = [(-6, 20), (3, 15), (2, 19), (1, 12)]
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for a, title in zip(ax, ["Gradient descent", "Stochastic gradient descent (batch 7)"]):
    a.contourf(A0, A1, LG, 30, cmap="viridis"); a.set_title(title); a.set_xlabel(r"$\phi_0$")
for s, col in zip(starts, [TB["orange"], "w", TB["red"], TB["lblue"]]):
    p = gd(gabor_grad, s, 0.01, 2000); ax[0].plot(p[:, 0], p[:, 1], ".-", color=col, ms=2, lw=1)
    q = sgd_gabor(s, 0.2, 2000, seed=3); ax[1].plot(q[:, 0], q[:, 1], ".-", color=col, ms=2, lw=0.8)
    print(f"start {s}: GD ends at loss {gabor_loss(p[-1]):6.3f}   SGD ends at loss {gabor_loss(q[-1]):6.3f}")
for a in ax: a.plot(0, 16.6, "*", color="w", ms=14)
ax[0].set_ylabel(r"$\phi_1$"); plt.show()
esc = sum(gabor_loss(sgd_gabor((3, 15), 0.2, 2000, seed=k)[-1]) < 0.5 for k in range(20))
print(f"From (3, 15) GD always gets stuck. SGD reached the global minimum (loss ≈ 0.44) in {esc}/20 random runs.")
print("Lesson: in a 2-parameter landscape the starting point dominates; SGD *can* escape, but there's no guarantee.")

In [ ]:
#@title ⚙️ Ready-made helper: `momentum_weight` (just run me)
def momentum_weight(k, beta):
    return (1 - beta) * beta ** k

### 🧩 Puzzle 6.4 ⭐⭐ Implement momentum, Nesterov and Adam from the equations
Each optimiser takes `grad_fn(phi) -> gradient` and returns the **path**: an array of shape `(n_steps+1, n_params)` that includes the start. The checker compares your paths with reference implementations.

In [ ]:
def momentum(grad_fn, phi, alpha, beta=0.9, n_steps=100):
    # TODO
    raise NotImplementedError

def nesterov(grad_fn, phi, alpha, beta=0.9, n_steps=100):
    # TODO  (gradient evaluated at phi - alpha*beta*m)
    raise NotImplementedError

def adam(grad_fn, phi, alpha, beta=0.9, gamma=0.99, eps=1e-8, n_steps=100):
    # TODO  (use t+1 in the bias-correction exponents, t = 0, 1, 2, ...)
    raise NotImplementedError

In [ ]:
# 🧪 Check puzzle 6.4
def _t64():
    gf = lambda p: np.array([0.1 * p[0], 10 * p[1]]) + np.array([np.cos(p[1]), 0.0])
    p0 = np.array([3.0, 1.0])
    # reference trajectories (computed independently)
    m = np.zeros(2); ph = p0.copy(); R1 = [ph.copy()]
    for _ in range(50): m = .9 * m + .1 * gf(ph); ph = ph - .05 * m; R1.append(ph.copy())
    m = np.zeros(2); ph = p0.copy(); R2 = [ph.copy()]
    for _ in range(50): m = .9 * m + .1 * gf(ph - .05 * .9 * m); ph = ph - .05 * m; R2.append(ph.copy())
    m = np.zeros(2); v = np.zeros(2); ph = p0.copy(); R3 = [ph.copy()]
    for t in range(50):
        g = gf(ph); m = .9 * m + .1 * g; v = .99 * v + .01 * g ** 2
        ph = ph - .05 * (m / (1 - .9 ** (t + 1))) / (np.sqrt(v / (1 - .99 ** (t + 1))) + 1e-8); R3.append(ph.copy())
    ok = [close(momentum(gf, p0, .05, .9, 50), R1, 1e-10), close(nesterov(gf, p0, .05, .9, 50), R2, 1e-10),
          close(adam(gf, p0, .05, .9, .99, 1e-8, 50), R3, 1e-10)]
    print("   momentum / nesterov / adam:", ["✅" if o else "❌" for o in ok])
    return all(ok)
check("6.4", 2, _t64, "Three optimisers in ~15 lines. torch.optim does exactly this.",
      "Path must include the starting point and have n_steps+1 rows.")

In [ ]:
#@title 🔑 Solution 6.4 — double-click to peek (have a go first!)
peek('6.4')
def momentum(grad_fn, phi, alpha, beta=0.9, n_steps=100):
    phi = np.array(phi, float); m = np.zeros_like(phi); path = [phi.copy()]
    for _ in range(n_steps):
        m = beta * m + (1 - beta) * grad_fn(phi); phi = phi - alpha * m; path.append(phi.copy())
    return np.array(path)

def nesterov(grad_fn, phi, alpha, beta=0.9, n_steps=100):
    phi = np.array(phi, float); m = np.zeros_like(phi); path = [phi.copy()]
    for _ in range(n_steps):
        m = beta * m + (1 - beta) * grad_fn(phi - alpha * beta * m); phi = phi - alpha * m; path.append(phi.copy())
    return np.array(path)

def adam(grad_fn, phi, alpha, beta=0.9, gamma=0.99, eps=1e-8, n_steps=100):
    phi = np.array(phi, float); m = np.zeros_like(phi); v = np.zeros_like(phi); path = [phi.copy()]
    for t in range(n_steps):
        g = grad_fn(phi); m = beta * m + (1 - beta) * g; v = gamma * v + (1 - gamma) * g ** 2
        mt = m / (1 - beta ** (t + 1)); vt = v / (1 - gamma ** (t + 1))
        phi = phi - alpha * mt / (np.sqrt(vt) + eps); path.append(phi.copy())
    return np.array(path)

In [ ]:
# 💻 Compare them on a long, narrow valley (Fig 6.9): steep in phi1, shallow in phi0
valley = lambda p: 0.5 * (0.05 * p[0] ** 2 + 5 * p[1] ** 2)
valley_grad = lambda p: np.array([0.05 * p[0], 5 * p[1]])
start = [-9.0, 1.5]
runs = {"GD (α=0.38)": gd(valley_grad, start, 0.38, 80), "momentum": momentum(valley_grad, start, 0.38, 0.9, 80),
        "Nesterov": nesterov(valley_grad, start, 0.38, 0.9, 80), "Adam": adam(valley_grad, start, 0.6, 0.9, 0.99, n_steps=80)}
V0, V1 = np.meshgrid(np.linspace(-10, 2, 200), np.linspace(-2, 2, 200))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].contour(V0, V1, valley([V0, V1]), 25, cmap="Greys", linewidths=0.6)
for name, p in runs.items():
    ax[0].plot(p[:, 0], p[:, 1], ".-", ms=3, lw=1, label=name)
    ax[1].semilogy([valley(q) + 1e-12 for q in p], label=name)
ax[0].legend(); ax[0].set_title("Paths"); ax[1].set_title("Loss vs step"); ax[1].legend(); plt.show()

### 🏆 Challenge 6 ⭐⭐ Race to the burrow
The **dune function** $L(a,b)=(1-a)^2+20\,(b-a^2)^2$ (a Rosenbrock cousin) has a curved, banana-shaped valley. The burrow is at $(1,1)$ and you start at $(-1.5,\,2)$.

Pick an optimiser and its hyperparameters. You win when the loss drops below $10^{-4}$, and your score is the **number of steps** it took (maximum 3,000). Fewer steps is better.
- 🥉 < 2000 steps
- 🥈 < 600 steps
- 🥇 < 200 steps

In [ ]:
dune = lambda p: (1 - p[0]) ** 2 + 20 * (p[1] - p[0] ** 2) ** 2
dune_grad = lambda p: np.array([-2 * (1 - p[0]) - 80 * p[0] * (p[1] - p[0] ** 2), 40 * (p[1] - p[0] ** 2)])

def race(path):
    L = np.array([dune(p) for p in path]); hit = np.flatnonzero(L < 1e-4)
    steps = int(hit[0]) if len(hit) else 10 ** 9
    D0, D1 = np.meshgrid(np.linspace(-2, 2, 300), np.linspace(-1, 3, 300))
    plt.figure(figsize=(5, 4)); plt.contour(D0, D1, np.log(dune([D0, D1]) + 1e-3), 30, cmap="YlOrBr")
    plt.plot(path[:, 0], path[:, 1], ".-", color=TB["green"], ms=2, lw=0.8); plt.plot(1, 1, "*", color=TB["red"], ms=15)
    plt.title(f"steps to burrow: {steps if steps < 10**9 else 'DNF'}"); plt.show()
    return steps

In [ ]:
my_path = gd(dune_grad, [-1.5, 2.0], lr=0.001, n=3000)   # TODO: swap in momentum / nesterov / adam and tune!

In [ ]:
# 🧪 Check puzzle C6
def _tc6():
    assert len(my_path) <= 3001 and np.allclose(my_path[0], [-1.5, 2.0]), "start at (-1.5, 2) and use <= 3000 steps"
    s = race(np.asarray(my_path)); print(f"   {s if s < 10**9 else 'did not finish'} steps → {medal(s, 2000, 600, 200)}")
    return s < 200
check("C6", 2, _tc6, "First into the burrow. The mob is impressed.",
      "Try adam(dune_grad, [-1.5, 2.0], alpha=0.05..0.3, ...) or momentum with beta ≈ 0.9 and a bigger lr.")

In [ ]:
#@title 🔑 Solution C6 — double-click to peek (have a go first!)
peek('C6')
my_path = adam(dune_grad, [-1.5, 2.0], alpha=0.2, beta=0.9, gamma=0.99, n_steps=3000)

---
# Part 7 · Gradients and initialization (Chapter 7): 📣 whispers down the burrow
### 📖 Recap: backpropagation = chain rule + caching
**Toy model** (eq 7.5): one hidden unit per layer, with sin, exp and cos as the activations:
$$\mathrm{f}[x,\boldsymbol\phi]=\beta_3+\omega_3\cos\!\Big[\beta_2+\omega_2\exp\big[\beta_1+\omega_1\sin[\beta_0+\omega_0x]\big]\Big],\qquad \ell_i=(\mathrm{f}[x_i,\boldsymbol\phi]-y_i)^2 .$$

**Forward pass** (eq 7.9): compute and *store* $f_0=\beta_0+\omega_0x,\ h_1=\sin f_0,\ f_1=\beta_1+\omega_1h_1,\ h_2=\exp f_1,\ f_2=\beta_2+\omega_2h_2,\ h_3=\cos f_2,\ f_3=\beta_3+\omega_3h_3$.

**Backward pass #1**: work backwards through the intermediate variables. $\frac{\partial\ell}{\partial f_3}=2(f_3-y)$, then $\frac{\partial\ell}{\partial h_3}=\omega_3\frac{\partial\ell}{\partial f_3}$, then $\frac{\partial\ell}{\partial f_2}=-\sin f_2\,\frac{\partial\ell}{\partial h_3}$, and so on.

**Backward pass #2**: the parameter gradients are $\frac{\partial\ell}{\partial\beta_k}=\frac{\partial\ell}{\partial f_k}$ and $\frac{\partial\ell}{\partial\omega_k}=h_k\frac{\partial\ell}{\partial f_k}$, with $h_0=x$.

**Vector form** (eqs 7.24–7.26), for a deep ReLU net:
$$\frac{\partial\ell}{\partial\boldsymbol\beta_k}=\frac{\partial\ell}{\partial\mathbf{f}_k},\qquad \frac{\partial\ell}{\partial\boldsymbol\Omega_k}=\frac{\partial\ell}{\partial\mathbf{f}_k}\mathbf{h}_k^T,\qquad \frac{\partial\ell}{\partial\mathbf{f}_{k-1}}=\mathbb{I}[\mathbf{f}_{k-1}>0]\odot\Big(\boldsymbol\Omega_k^T\frac{\partial\ell}{\partial\mathbf{f}_k}\Big).$$

In [ ]:
# 💻 Toy model forward pass
TOY = dict(beta0=1.0, beta1=2.0, beta2=-0.5, beta3=0.4, omega0=0.1, omega1=-0.4, omega2=2.0, omega3=3.0)
x_toy, y_toy = 2.3, 2.0

def toy_forward(x, p):
    f0 = p["beta0"] + p["omega0"] * x; h1 = np.sin(f0)
    f1 = p["beta1"] + p["omega1"] * h1; h2 = np.exp(f1)
    f2 = p["beta2"] + p["omega2"] * h2; h3 = np.cos(f2)
    f3 = p["beta3"] + p["omega3"] * h3
    return dict(f0=f0, h1=h1, f1=f1, h2=h2, f2=f2, h3=h3, f3=f3)

def toy_loss(x, y, p):
    return (toy_forward(x, p)["f3"] - y) ** 2
print("forward values:", {k: round(v, 4) for k, v in toy_forward(x_toy, TOY).items()}, "| loss", round(toy_loss(x_toy, y_toy, TOY), 4))

### 🧩 Puzzle 7.1 ⭐⭐ Backprop the toy model by hand
Fill in the backward pass. Every line is one multiplication by a local derivative.

In [ ]:
def toy_backward(x, y, p):
    v = toy_forward(x, p)
    # Backward pass #1: derivatives w.r.t. intermediate quantities
    dl_df3 = 2 * (v["f3"] - y)
    dl_dh3 = None   # TODO
    dl_df2 = None   # TODO
    dl_dh2 = None   # TODO
    dl_df1 = None   # TODO
    dl_dh1 = None   # TODO
    dl_df0 = None   # TODO
    # Backward pass #2: derivatives w.r.t. parameters
    grads = dict(beta0=None, omega0=None, beta1=None, omega1=None,      # TODO
                 beta2=None, omega2=None, beta3=None, omega3=None)
    return grads

In [ ]:
# 🧪 Check puzzle 7.1
def _t71():
    g = toy_backward(x_toy, y_toy, TOY)
    if any(v is None for v in g.values()): raise NotImplementedError
    keys = list(TOY)
    def L(vec): return toy_loss(x_toy, y_toy, dict(zip(keys, vec)))
    gn = num_grad(L, [TOY[k] for k in keys])
    bad = [k for k, n in zip(keys, gn) if not np.isclose(g[k], n, rtol=1e-5, atol=1e-6)]
    if bad: print("   mismatch:", bad)
    return not bad
check("7.1", 2, _t71, "Notice how each line reuses the one before it. That reuse is the whole trick.",
      "d sin = cos, d exp = exp (= h2!), d cos = -sin. And dl/domega_k = h_k * dl/df_k, with h_0 = x.")

In [ ]:
#@title 🔑 Solution 7.1 — double-click to peek (have a go first!)
peek('7.1')
def toy_backward(x, y, p):
    v = toy_forward(x, p)
    dl_df3 = 2 * (v["f3"] - y)
    dl_dh3 = p["omega3"] * dl_df3
    dl_df2 = -np.sin(v["f2"]) * dl_dh3
    dl_dh2 = p["omega2"] * dl_df2
    dl_df1 = v["h2"] * dl_dh2            # derivative of exp is exp
    dl_dh1 = p["omega1"] * dl_df1
    dl_df0 = np.cos(v["f0"]) * dl_dh1
    return dict(beta0=dl_df0, omega0=x * dl_df0, beta1=dl_df1, omega1=v["h1"] * dl_df1,
                beta2=dl_df2, omega2=v["h2"] * dl_df2, beta3=dl_df3, omega3=v["h3"] * dl_df3)

In [ ]:
#@title ⚙️ Ready-made helper: `deep_backward` (just run me)
def deep_backward(all_f, all_h, omegas, y):
    K = len(omegas) - 1
    dl_dbetas, dl_domegas = [None] * (K + 1), [None] * (K + 1)
    dl_df = 2 * (all_f[-1] - y)
    for k in range(K, -1, -1):
        dl_dbetas[k] = dl_df.sum(axis=1, keepdims=True)
        dl_domegas[k] = dl_df @ all_h[k].T
        if k > 0:
            dl_df = (all_f[k - 1] > 0) * (omegas[k].T @ dl_df)
    return dl_dbetas, dl_domegas

### 📖 Initialization (§7.5): keep the whisper audible
A message whispered down a 50-meerkat burrow either fades to nothing or turns into a scream. Activations and gradients behave the same way as they pass through layers.

With zero biases and weights $\Omega_{ij}\sim\mathcal{N}(0,\sigma^2_\Omega)$, independent of the activations:
$$\mathbb{E}[f'_i]=0,\qquad \sigma^2_{f'}=\sigma^2_\Omega\sum_{j=1}^{D_h}\mathbb{E}[h_j^2]=\sigma^2_\Omega D_h\,\mathbb{E}[h^2]. \qquad\text{(eq 7.30)}$$
If $f$ is symmetric about zero, then the ReLU gives $\mathbb{E}[h^2]=\tfrac12\sigma_f^2$, so $\sigma^2_{f'}=\tfrac12D_h\sigma^2_\Omega\,\sigma_f^2$ (eq 7.31). Choosing
$$\boxed{\sigma^2_\Omega=\frac{2}{D_h}}\qquad\textbf{(He initialization, eq 7.32)}$$
keeps the variance constant from layer to layer. In the backward pass the same argument uses $D_{h'}$ (eq 7.33), and a compromise between the two is $\sigma^2_\Omega=\frac{4}{D_h+D_{h'}}$ (eq 7.34).

In [ ]:
#@title ⚙️ Ready-made helper: `relu_moments` (just run me)
def relu_moments(sigma):
    return sigma / np.sqrt(2 * np.pi), sigma ** 2 / 2

In [ ]:
# 💻 Recreate Fig 7.7: 50 layers x 100 units, five weight variances
def init_experiment(sigma2, K=50, D=100, seed=0):
    rr = np.random.default_rng(seed)
    betas, omegas = init_params([D] + [D] * K + [1], sigma2=sigma2, rng=rr)
    xin = rr.normal(size=(D, 1))
    out, fs, hs = deep_forward(xin, betas, omegas)
    db, dO = deep_backward(fs, hs, omegas, np.zeros((1, 1)))
    fwd = [np.var(h) for h in hs[1:]]
    bwd = [np.var(g) for g in dO[:-1]]
    return fwd, bwd

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for s2, col in zip([0.001, 0.01, 0.02, 0.1, 1.0], [TB["blue"], TB["lblue"], TB["green"], TB["orange"], TB["red"]]):
    fwd, bwd = init_experiment(s2)
    ax[0].semilogy(fwd, color=col, lw=2, label=fr"$\sigma^2_\Omega$={s2}")
    ax[1].semilogy(bwd, color=col, lw=2, label=fr"$\sigma^2_\Omega$={s2}")
ax[0].set_title("forward: variance of activations"); ax[0].set_xlabel("layer")
ax[1].set_title("backward: variance of weight gradients"); ax[1].set_xlabel("layer")
ax[0].legend(); plt.show()
print("Only σ² = 2/D_h = 0.02 (He) keeps both curves flat. Everything else vanishes or explodes exponentially.")

In [ ]:
#@title ⚙️ Ready-made helper: `predicted_preact_var` (just run me)
def predicted_preact_var(k, sigma2, Di, Dh):
    return Di * sigma2 * (Dh * sigma2 / 2) ** k

### 💻 The book's training loop (Fig 7.8), in PyTorch
Everything we built by hand now fits in about 20 lines: He initialization, an MSE loss, SGD with momentum, a step learning-rate schedule, and `loss.backward()` doing Puzzle 7.2 for us.

In [ ]:
if HAS_TORCH:
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
    torch.manual_seed(0)
    D_i, D_k, D_o = 10, 40, 5
    model = nn.Sequential(nn.Linear(D_i, D_k), nn.ReLU(), nn.Linear(D_k, D_k), nn.ReLU(), nn.Linear(D_k, D_o))
    def weights_init(layer_in):                                # He initialization
        if isinstance(layer_in, nn.Linear):
            nn.init.kaiming_normal_(layer_in.weight); layer_in.bias.data.fill_(0.0)
    model.apply(weights_init)
    criterion = nn.MSELoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
    x = torch.randn(100, D_i); y = torch.randn(100, D_o)       # random data, as in the book
    data_loader = DataLoader(TensorDataset(x, y), batch_size=10, shuffle=True)
    hist = []
    for epoch in range(100):
        epoch_loss = 0.0
        for x_batch, y_batch in data_loader:
            optimizer.zero_grad(); loss = criterion(model(x_batch), y_batch)
            loss.backward(); optimizer.step(); epoch_loss += loss.item()
        hist.append(epoch_loss); scheduler.step()
    plt.semilogy(hist, color=TB["green"]); plt.xlabel("epoch"); plt.ylabel("training loss")
    plt.title("Fitting 100 random targets (lr 0.01, halved every 10 epochs)"); plt.show()

---
# Part 8 · Measuring performance (Chapter 8): 🎙️ the meerkat call classifier
### 📖 Recap
- **Generalization** is measured on a separate **test set**. Training error can reach zero while test error stays high.
- For least squares regression, the expected test error splits *exactly* into three parts (eq 8.7):
$$\mathbb{E}_{\mathcal{D}}\Big[\mathbb{E}_y\big[L[x]\big]\Big]=\underbrace{\mathbb{E}_{\mathcal{D}}\Big[\big(\mathrm{f}[x,\boldsymbol\phi[\mathcal{D}]]-\mathrm{f}_\mu[x]\big)^2\Big]}_{\text{variance}}+\underbrace{\big(\mathrm{f}_\mu[x]-\mu[x]\big)^2}_{\text{bias}^2}+\underbrace{\sigma^2}_{\text{noise}}$$
  where $\mathrm{f}_\mu[x]=\mathbb{E}_{\mathcal{D}}\big[\mathrm{f}[x,\boldsymbol\phi[\mathcal{D}]]\big]$ is the average fitted model over training sets.
- **Variance** falls with more data. **Bias** falls with more capacity, which usually raises variance: the **bias–variance trade-off**.
- **Double descent**: as capacity grows past the point where the model can memorise the training data, test error can fall *again*.
- The **curse of dimensionality** means high-dimensional training data are extremely sparse, so behaviour *between* data points (the inductive bias) matters.
- Choose **hyperparameters** on a **validation** set, and touch the test set only once.

#### Derivation sketch (eqs 8.2–8.3)
Add and subtract $\mu[x]$: $\;(\mathrm{f}-y)^2=(\mathrm{f}-\mu)^2+2(\mathrm{f}-\mu)(\mu-y)+(\mu-y)^2$. Taking $\mathbb{E}_y$ kills the cross term because $\mathbb{E}_y[y]=\mu$. Repeat the trick with $\mathrm{f}_\mu$ and $\mathbb{E}_{\mathcal{D}}$ to split $(\mathrm{f}-\mu)^2$ into variance plus bias².

In [ ]:
# 💻 The meerkat call dataset: an MNIST-1D-style benchmark (Greydanus 2020), meerkat edition.
CALLS = ["eagle alarm", "jackal alarm", "snake alarm", "close call", "lead call",
         "moving call", "sentinel song", "recruitment", "pup begging", "social chirp"]

def make_meerkat_calls(n, seed=0, noise=0.75, shift=12):
    tr = np.random.default_rng(1234)                             # fixed templates (same species, same repertoire!)
    templates = np.cumsum(tr.normal(0, 1, (10, 12)), axis=1); templates -= templates.mean(1, keepdims=True)
    templates /= np.abs(templates).max(1, keepdims=True)
    r = np.random.default_rng(seed); y = r.integers(0, 10, n); X = np.zeros((n, 40))
    t40 = np.linspace(0, 1, 40)
    for i in range(n):
        tpl = templates[y[i]]; stretch = r.uniform(0.75, 1.25)
        tt = np.clip((t40 - 0.5) * stretch + 0.5, 0, 1)
        sig = np.interp(tt, np.linspace(0, 1, 12), tpl) * r.uniform(0.7, 1.3)
        sig = np.roll(sig, r.integers(-shift, shift + 1))
        sig += r.uniform(-0.6, 0.6) * (t40 - 0.5)                                 # slope (background wind)
        sig += noise * np.convolve(r.normal(size=44), np.ones(5) / 5, "valid")    # correlated noise
        X[i] = sig + 0.15 * noise * r.normal(size=40)
    return X.astype(np.float32), y

Xtr, ytr = make_meerkat_calls(4000, seed=1); Xte, yte = make_meerkat_calls(1000, seed=2)
fig, axs = plt.subplots(2, 5, figsize=(13, 4), sharex=True, sharey=True)
for k, ax in enumerate(axs.ravel()):
    for j in np.flatnonzero(ytr == k)[:4]: ax.plot(Xtr[j], lw=1)
    ax.set_title(CALLS[k], fontsize=9)
plt.suptitle("Four examples of each call type (40 samples each)", fontweight="bold"); plt.tight_layout(); plt.show()

In [ ]:
# 💻 Train the §8.1 model: 40 → 100 → 100 → 10, SGD, lr 0.1, batch 100, 6000 steps
def train_mlp(Xtr, ytr, Xev, yev, width=100, depth=2, lr=0.1, steps=6000, batch=100, seed=0, opt="sgd", log_every=100):
    torch.manual_seed(seed)
    layers, d = [], Xtr.shape[1]
    for _ in range(depth): layers += [torch.nn.Linear(d, width), torch.nn.ReLU()]; d = width
    net = torch.nn.Sequential(*layers, torch.nn.Linear(d, 10))
    for m in net:
        if isinstance(m, torch.nn.Linear): torch.nn.init.kaiming_normal_(m.weight); m.bias.data.zero_()
    o = torch.optim.SGD(net.parameters(), lr=lr) if opt == "sgd" else torch.optim.Adam(net.parameters(), lr=lr)
    Xt, yt, Xe, ye = map(torch.tensor, (Xtr, ytr, Xev, yev)); lossf = torch.nn.CrossEntropyLoss()
    g = torch.Generator().manual_seed(seed); hist = []
    for s in range(steps + 1):
        if s % log_every == 0:
            with torch.no_grad():
                ot, oe = net(Xt), net(Xe)
                hist.append((s, (ot.argmax(1) != yt).float().mean().item(), (oe.argmax(1) != ye).float().mean().item(),
                             lossf(ot, yt).item(), lossf(oe, ye).item()))
        if s == steps: break
        idx = torch.randint(0, len(Xt), (batch,), generator=g)
        o.zero_grad(); lossf(net(Xt[idx]), yt[idx]).backward(); o.step()
    return net, np.array(hist)

if HAS_TORCH:
    t0 = time.time(); _, H = train_mlp(Xtr, ytr, Xte, yte); print(f"trained in {time.time()-t0:.1f}s")
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(H[:, 0], 100 * H[:, 1], label="train", color=TB["green"]); ax[0].plot(H[:, 0], 100 * H[:, 2], label="test", color=TB["orange"])
    ax[0].set_ylabel("% error"); ax[0].set_xlabel("step"); ax[0].legend(); ax[0].set_title("Error (cf. Fig 8.2a)")
    ax[1].plot(H[:, 0], H[:, 3], label="train", color=TB["green"]); ax[1].plot(H[:, 0], H[:, 4], label="test", color=TB["orange"])
    ax[1].set_xlabel("step"); ax[1].legend(); ax[1].set_title("Loss (cf. Fig 8.2b)"); plt.show()
    print(f"final train error {100*H[-1,1]:.1f}%   test error {100*H[-1,2]:.1f}%   (chance = 90%)")

In [ ]:
# 🧩 Puzzle 8.1 ⭐  The test LOSS climbs late in training while the test ERROR stays flat. Why?
#  (a) the learning rate is too high, so the optimiser diverges
#  (b) the model keeps making the same mistakes but with growing confidence (softmax logits pushed to extremes)
#  (c) the test set is mislabelled
#  (d) ReLUs die during training
answer_8_1 = None   # TODO "a" / "b" / "c" / "d"

In [ ]:
# 🧪 Check puzzle 8.1
def _t81():
    if answer_8_1 is None: raise NotImplementedError
    return answer_8_1.strip().lower() == "b"
check("8.1", 1, _t81, "Confidently wrong predictions get punished by the log in the cross-entropy (see Part 5B).")

In [ ]:
#@title 🔑 Solution 8.1 — double-click to peek (have a go first!)
peek('8.1')
answer_8_1 = "b"

### 📖 Noise, bias and variance, measured exactly (§8.2)
To *measure* the three terms we need to know the truth, so we go back to 1D regression. The true function is $\mu[x]=\exp(\sin(2\pi x))$, the noise has $\sigma=0.3$, and each dataset has 15 points (one per equal segment of $[0,1]$). The **simplified network** of Fig 8.4 fixes the joints at $0,\frac1D,\dots,\frac{D-1}{D}$, so only the output layer is learned. That makes it **linear in its parameters**, so we can fit it in closed form (book P8.3):
$$\mathbf{A}=\big[\mathbf{1},\ \mathrm{ReLU}[\mathbf{x}-0],\ \mathrm{ReLU}[\mathbf{x}-\tfrac1D],\dots\big],\qquad \hat{\boldsymbol\phi}=(\mathbf{A}^T\mathbf{A})^{-1}\mathbf{A}^T\mathbf{y}.$$

In [ ]:
def mu_true(x): return np.exp(np.sin(2 * np.pi * x))
SIGMA = 0.3
def make_reg_data(n, r):
    x = (np.arange(n) + r.uniform(size=n)) / n          # one sample per equal segment
    return x, mu_true(x) + r.normal(0, SIGMA, n)

In [ ]:
#@title ⚙️ Ready-made helper: `design`, `fit_fixed_joints` (just run me)
def design(x, D):
    return np.column_stack([np.ones_like(x)] + [relu(x - k / D) for k in range(D)])
def fit_fixed_joints(x, y, D):
    return np.linalg.lstsq(design(x, D), y, rcond=None)[0]

In [ ]:
#@title ⚙️ Ready-made helper: `bias_variance` (just run me)
def bias_variance(D, n_datasets=300, n_points=15, seed=0):
    r = np.random.default_rng(seed); xg = np.linspace(0, 1, 200)
    fits = np.array([design(xg, D) @ fit_fixed_joints(*make_reg_data(n_points, r), D) for _ in range(n_datasets)])
    f_mu = fits.mean(0)
    return np.mean((f_mu - mu_true(xg)) ** 2), np.mean(fits.var(0))

In [ ]:
# 💻 The bias-variance trade-off (Fig 8.9)
Ds = np.arange(1, 13); bv = np.array([bias_variance(D, 300, 15) for D in Ds])
plt.plot(Ds, bv[:, 0], "o-", color=TB["orange"], label="bias²"); plt.plot(Ds, bv[:, 1], "o-", color=TB["blue"], label="variance")
plt.plot(Ds, bv.sum(1), "--", color=TB["grey"], label="bias² + variance")
best = Ds[np.argmin(bv.sum(1))]; plt.axvline(best, color=TB["green"], ls=":", label=f"best capacity D={best}")
plt.yscale("log"); plt.xlabel("hidden units D (= linear regions)"); plt.legend(); plt.title("Bias-variance trade-off, 15 points"); plt.show()

### 📖 Double descent (§8.4)
Below we use a shallow net with a **random, frozen** first layer (random ReLU features) and fit the output weights with the **minimum-norm** least squares solution (`np.linalg.lstsq`). Minimum norm is a simple stand-in for the implicit regularisation the book suspects SGD provides. We have 30 noisy training points and sweep the width from 1 to 2,000 hidden units, tracking both training and test error. We call the width where training MSE first drops below $\sigma^2/10$ the **(near-)interpolation threshold**: past it, the model has essentially memorised the noise.

In [ ]:
def random_feature_curve(widths, n_train=30, n_test=500, n_rep=40, seed=0):
    r = np.random.default_rng(seed); te = np.zeros((n_rep, len(widths))); tr = np.zeros_like(te)
    for rep in range(n_rep):
        xtr, ytr_ = make_reg_data(n_train, r); xts = r.uniform(size=n_test); yts = mu_true(xts) + r.normal(0, SIGMA, n_test)
        c = r.uniform(-0.1, 1.1, 2000)                                        # joint positions
        a = np.abs(r.normal(0, 1, 2000)) * r.choice([-1, 1], 2000); b = -a * c # frozen first layer
        for j, D in enumerate(widths):
            feats = lambda x: np.column_stack([np.ones_like(x), relu(a[:D] * x[:, None] + b[:D])])
            w = np.linalg.lstsq(feats(xtr), ytr_, rcond=None)[0]               # minimum-norm when under-determined
            te[rep, j] = np.mean((feats(xts) @ w - yts) ** 2); tr[rep, j] = np.mean((feats(xtr) @ w - ytr_) ** 2)
    return np.median(tr, 0), np.median(te, 0)

widths = np.unique(np.round(np.logspace(0, np.log10(2000), 45)).astype(int))
tr_err, te_err = random_feature_curve(widths)
D_interp = widths[np.argmax(tr_err < 0.1 * SIGMA ** 2)]       # first width that (almost) memorises the training set
plt.loglog(widths, te_err, "o-", color=TB["purple"], ms=4, label="test")
plt.loglog(widths, np.maximum(tr_err, 1e-4), "o-", color=TB["green"], ms=3, label="train (floored at 1e-4)")
plt.axvline(D_interp, color=TB["grey"], ls=":"); plt.text(D_interp * 1.15, 2e-3, "(near-)interpolation\nthreshold", color=TB["grey"])
plt.axhline(SIGMA ** 2, color=TB["orange"], ls="--", lw=1); plt.text(150, SIGMA ** 2 * 0.62, "noise floor σ²", color=TB["orange"])
plt.xlabel("hidden units D"); plt.ylabel("median MSE"); plt.legend(loc="lower left"); plt.title("Double descent with random ReLU features"); plt.show()
print(f"training MSE first drops below σ²/10 at D = {D_interp} | test-error peak at D = {widths[np.argmax(te_err)]}")
print(f"classical sweet spot: {te_err[widths < D_interp].min():.3f} | peak: {te_err.max():.2f} | D=2000: {te_err[-1]:.3f}")
print("In this tiny 1D problem the second descent recovers but doesn't beat the classical sweet spot.\n"
      "On MNIST-1D with label noise (Fig 8.10b) it does. How much it helps depends on data, noise and inductive bias.")

In [ ]:
# 🧩 Puzzle 8.4 ⭐  Look at the plot. The test-error peak sits...
#  (a) at D = 1, where the model is too simple
#  (b) around the width where the model can first (almost) memorise the training data (the interpolation threshold)
#  (c) at the largest width, where there are the most parameters
#  (d) at D = 30 exactly, because there are 30 data points (no more, no less)
answer_8_4 = None   # TODO

In [ ]:
# 🧪 Check puzzle 8.4
def _t84():
    if answer_8_4 is None: raise NotImplementedError
    return answer_8_4.strip().lower() == "b"
check("8.4", 1, _t84, "Not exactly 30 here: random joints rarely land in every gap between data points, so the net needs more units than data points to memorise them.",
      "Compare the dotted line with the purple peak.")

In [ ]:
#@title 🔑 Solution 8.4 — double-click to peek (have a go first!)
peek('8.4')
answer_8_4 = "b"

In [ ]:
# 💻 Distances concentrate in high dimensions: nearest and farthest neighbours look alike
r = np.random.default_rng(0); dims = [1, 2, 5, 10, 40, 100, 1000]; ratios = []
for d in dims:
    P = r.uniform(size=(1000, d)); q = r.uniform(size=d); dist = np.linalg.norm(P - q, axis=1)
    ratios.append(dist.max() / dist.min())
plt.semilogx(dims, ratios, "o-", color=TB["blue"]); plt.yscale("log")
plt.xlabel("dimension"); plt.ylabel("farthest / nearest distance"); plt.title("Everything is far from everything"); plt.show()

---
# 🏁 Your progress

In [ ]:
progress()

### ⏭️ Next time
**Next up · Chapter 9 (Regularization):** why do over-parameterised networks generalise at all?

### 📚 Where to go next
- The book's own notebooks ([udlbook GitHub](https://github.com/udlbook/udlbook)) mirror every section here. Try Notebooks 3.3 (regions), 6.5 (Adam), 7.2 (backprop) and 8.3 (double descent).
- MNIST-1D (Greydanus, 2020) is the real version of our meerkat call dataset: `pip install mnist1d`.
- Discussion prompts for the group:
  1. A deep network is a flexible classifier. Where would *bias* versus *variance* dominate on real phase-separation data with ~100 measured proteins?
  2. Which output distribution (Part 5) suits a *condensate count per cell*? A *droplet size*? A *partition coefficient*?
  3. Double descent says "bigger can be better". When is that **not** true for our assay data?

*Built for the Transition Bio ML reading group · based on Prince, Understanding Deep Learning (MIT Press, CC-BY-NC-ND) · all data synthetic.*